# Use Subqueries with fda food BigQuery Public Dataset

## Activity Overview

As a data analyst, we will sometimes want to analyze a small subset of data that is contained within a much larger dataset. This is when subqueries can be really useful. As we know, subqueries are queries that are nested inside of another query. Some queries can have many subqueries, while others may have only one. It’s important to know how subqueries function and to understand the different components we can use to create them. 

In this notebook, we will work with queries and subqueries to examine a public health dataset. We will create subqueries to discover which industries receive an unusual number of complaints and, more importantly, which industries are connected to serious health issues. This will allow us to allocate our resources more effectively to safeguard public health.

## Scenario

In this scenario, we are a junior data analyst for a multinational food and beverage manufacturer. We and our team are responsible for maintaining the safety of a wide array of food products. Because of the overwhelming number of products on the market, we have been asked to prioritize which products need to be reviewed by your stakeholders.

While it's useful to know which food industries receive the most complaints, the more critical aspect to consider is identifying the complaints that lead to severe health consequences, such as hospital visits.

To complete this task, we'll analyze food event reports for targeted health interventions.

## Step 1: Access the Public Dataset
The first step is to access the public dataset. For this activity, we will use a BigQuery public dataset titled <b>fda_food</b>, which contains details about food recalls, consumer reactions, product descriptions, and product distribution and quantities. 

In [1]:
from google.cloud import bigquery

print(bigquery.__version__)

3.40.1


In [2]:
client = bigquery.Client()

print(client.project)

myproject001-504709


## Step 2: Gather an Initial Overview
To get an initial overview, let's run a SQL query to identify which food industries have the highest number of complaints. This will provide an initial dataset to work from. 

The query will group the product industries by name so the report results will fall under each industry title.\
ORDER BY and DESC will tell the query to order the output by count_reports in descending order so, the industry with the most amount of reports will appear at the top of the output table. \
LIMIT will limit the results to ten industries with the highest numbers of reports.

In [3]:
query = """
SELECT 
products_industry_name, 
COUNT(report_number) AS count_reports
--SELECT is used to identify the product industries by name. COUNT will count the number of reports and label them as count_reports.
FROM bigquery-public-data.fda_food.food_events
GROUP BY products_industry_name
ORDER BY count_reports DESC
LIMIT 10;
"""
df = client.query(query).to_dataframe()
df

,products_industry_name,count_reports
0,Vit/Min/Prot/Unconv Diet(Human/Animal),96988
1,Cosmetics,83540
2,Nuts/Edible Seed,5832
3,Vegetables/Vegetable Products,5428
4,Soft Drink/Water,4171
5,Bakery Prod/Dough/Mix/Icing,4122
6,Fruit/Fruit Prod,3916
7,Fishery/Seafood Prod,3535
8,Cereal Prep/Breakfast Food,2669
9,Dietary Conventional Foods/Meal Replacements,2651


## Step 3: Determine the Number of Hospitalizations

With a list of the industries receiving the most complaints, our next step is to find out which of these complaints led to hospitalizations.\
Let's filter the initial list accordingly and focus solely on complaints that resulted in hospital visits. 

In [4]:
query = """
SELECT 
products_industry_name, 
COUNT(report_number) AS count_hospitalizations
FROM
bigquery-public-data.fda_food.food_events
WHERE products_industry_name IN
(SELECT 
products_industry_name
FROM 
bigquery-public-data.fda_food.food_events
GROUP BY products_industry_name
ORDER BY COUNT(report_number) DESC LIMIT 10)
AND outcomes LIKE '%Hospitalization%'
--The AND operator displays a record if all the conditions are TRUE.
--The LIKE operator is used in a WHERE clause to search for a specified pattern in a column.
GROUP BY products_industry_name
ORDER BY count_hospitalizations DESC;
"""
df = client.query(query).to_dataframe()
df

,products_industry_name,count_hospitalizations
0,Vit/Min/Prot/Unconv Diet(Human/Animal),25606
1,Cosmetics,10272
2,Dietary Conventional Foods/Meal Replacements,855
3,Fishery/Seafood Prod,487
4,Vegetables/Vegetable Products,413
5,Nuts/Edible Seed,399
6,Soft Drink/Water,377
7,Bakery Prod/Dough/Mix/Icing,264
8,Fruit/Fruit Prod,228
9,Cereal Prep/Breakfast Food,157


Great! Through this process, we discovered which industries receive an unusual number of complaints and, more importantly, which are connected to serious health issues. \
This knowledge allows us to allocate our resources more effectively to safeguard public health. 

We have created subqueries using SELECT statements with FROM, GROUP BY, WHERE, IN, AS, COUNT, and ORDER BY clauses, as well as AND and LIKE operators to search and analyze the query results relating to reports of food events by industry. 